# 02f Donor-Bound Tokenizer Generation

This notebook builds the donor-bound tokenizer for the rebuilt workflow.

This tokenizer keeps the donor residue, anomer, and donor carbon together as one token, while leaving the acceptor carbon as a separate token. For example, `Galb1` can stay together while `-4` remains separate.

**Main outputs**
- tokenizer files saved under `tokenizers/donor_bound/<setting_label>/`
- `vocab.json`
- `tokenizer_config_summary.json`
- optionally `inspection_preview.csv`


## Setup and user settings

The first code cell performs the small amount of bootstrap work that must happen before shared project helpers can be imported. It mounts Google Drive, ensures the public GitHub repository is available in the Colab runtime, and makes the `src` package importable.

The second code cell contains the values you may need to edit before running the notebook. Review `PROJECT_ROOT` first, then confirm the training split filename, the overwrite setting, and whether you want to save the inspection preview.

**Expected output**
- a Google Drive mount confirmation
- a repository bootstrap path under `/content/`
- the resolved training-data path, tokenizer output folder, and setting label

**How to interpret it**
- if the printed output folder is not the donor-bound folder you intended, correct the settings before training
- if the repository bootstrap step fails, the Colab runtime likely could not reach GitHub or reuse the local clone


In [ ]:
# Standard library imports used for the initial Colab bootstrap.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read project data and save outputs.
drive.mount('/content/drive')

# These public GitHub settings identify the repository that stores the notebook
# helpers. They normally do not need to change unless the project is moved to a
# different repository.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = Path('/content') / REPO_NAME

# The repository must exist locally before the notebook can import shared
# helper modules from the src package.
if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

# Add the repository root to the Python import path so the shared setup helper
# can be imported in the next cell.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

from src.notebook_setup import bootstrap_notebook_from_settings

print(f'Repository bootstrap directory: {REPO_DIR}')


In [ ]:
from pathlib import Path

from src.tokenizer_notebook_utils import build_tokenizer_output_paths, build_tokenizer_paths

# Update PROJECT_ROOT if your Drive project folder has a different name or
# location. This is the main path value that should be checked before running.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# This notebook trains on the training split created earlier in the workflow.
TRAIN_SPLIT_FILENAME = 'train.txt'

# If True, the notebook may replace previously saved tokenizer artifacts in the
# target output folder. If False, the notebook will stop before overwriting files.
OVERWRITE_EXISTING_OUTPUTS = False

# Save a lightweight CSV preview of the tokenizer sanity check when True.
SAVE_INSPECTION_PREVIEW = True

# This deterministic tokenizer does not learn merge settings, so the label
# simply records that the vocabulary came from the training split only.
SETTING_LABEL = 'v1_train_only'
TOKENIZER_FAMILY = 'donor_bound'

# Use the shared setup helper to update the local repository copy and validate
# the project root while keeping the editable settings visible in this notebook.
ctx = bootstrap_notebook_from_settings(
    notebook_name='02f_donor_bound_gen',
    project_root=PROJECT_ROOT,
    github_owner=GITHUB_OWNER,
    repo_name=REPO_NAME,
    github_ref=GITHUB_REF,
    repo_dir=REPO_DIR,
    require_drive=False,
    require_repo_sync=True,
)
project_root = ctx.project_root

# Build the standard paths used by this tokenizer notebook.
paths = build_tokenizer_paths(
    project_root=project_root,
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    train_split_filename=TRAIN_SPLIT_FILENAME,
)
output_paths = build_tokenizer_output_paths(
    tokenizer_output_dir=paths['tokenizer_output_dir'],
    include_inspection_preview=SAVE_INSPECTION_PREVIEW,
    include_merges_file=False,
)

train_data_path = paths['train_data_path']
tokenizer_output_dir = paths['tokenizer_output_dir']
config_summary_path = paths['config_summary_path']
inspection_preview_path = paths['inspection_preview_path']
vocab_path = paths['vocab_path']

tokenizer_output_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Training data path: {train_data_path}')
print(f'Tokenizer output directory: {tokenizer_output_dir}')
print(f'Setting label: {SETTING_LABEL}')


## Train, save, and inspect the tokenizer

The next code cell validates the input and output paths, builds the donor-bound vocabulary from the training split, saves the tokenizer files, and writes a short configuration summary JSON. The goal is to preserve both the learned token inventory and the exact tokenizer pattern used to apply it later.

The final code cell reloads the saved tokenizer from disk and shows a small inspection table built from a few training sequences. For this tokenizer, the preview is most useful when donor-side units such as `Galb1` stay together while acceptor-side units such as `-4` remain separate.

**Expected output**
- a confirmation that `vocab.json` was saved
- a reported vocabulary size and non-special token count
- a confirmation that tokenizer files and the summary JSON were saved
- a small inspection table with sample sequences, token counts, and token text

**How to interpret it**
- if donor information and acceptor information collapse into one token too often, the tokenizer is not preserving the intended split
- if the vocabulary is much smaller than expected, the training split or splitter function may not reflect the notation you intended to capture
- if the output folder is wrong, update `PROJECT_ROOT` or `SETTING_LABEL` before rerunning


In [ ]:
from src.notebook_utils import require_existing_path, validate_output_paths
from src.tokenizer_notebook_utils import (
    build_tokenizer_summary_payload,
    build_wordlevel_tokenizer_artifacts,
    load_training_sequences_for_tokenizer,
    save_tokenizer_summary,
)
from src.tokenizer_utils import SPECIAL_TOKENS, split_glycan_string_donor_bound

# Verify that the notebook can find the training split before continuing.
require_existing_path(train_data_path, 'Tokenizer training split')

# Enforce the shared overwrite policy before any tokenizer artifacts are saved.
validate_output_paths(
    output_paths=output_paths,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
)

# Load the training sequences that will define the fixed donor-bound vocabulary.
train_sequences = load_training_sequences_for_tokenizer(train_data_path)

# Build the vocabulary and saved tokenizer artifacts together so the tokenizer
# pattern stays aligned with the donor-bound units observed during vocab creation.
vocab, token_counts, PRETOKENIZER_PATTERN, saved_files = build_wordlevel_tokenizer_artifacts(
    train_sequences=train_sequences,
    tokenize_function=split_glycan_string_donor_bound,
    vocab_path=vocab_path,
    tokenizer_output_dir=tokenizer_output_dir,
)

print(f'Donor-bound vocabulary saved to: {vocab_path}')
print(f'Vocabulary size: {len(vocab)}')
print(f'Unique non-special tokens: {len(vocab) - len(SPECIAL_TOKENS)}')

# Record the saved vocabulary statistics and tokenizer pattern for later review.
tokenizer_summary = build_tokenizer_summary_payload(
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    train_data_path=train_data_path,
    tokenizer_output_dir=tokenizer_output_dir,
    saved_files=saved_files,
    extra_fields={
        'pretokenizer_pattern': PRETOKENIZER_PATTERN,
        'vocab_size': len(vocab),
        'num_special_tokens': len(SPECIAL_TOKENS),
        'num_non_special_tokens': len(vocab) - len(SPECIAL_TOKENS),
        'top_10_tokens': token_counts.most_common(10),
    },
)

save_tokenizer_summary(tokenizer_summary, config_summary_path)

print('Donor-bound tokenizer saved.')
print(f'Tokenizer folder: {tokenizer_output_dir}')


In [ ]:
from IPython.display import display
from transformers import PreTrainedTokenizerFast

from src.tokenizer_notebook_utils import (
    build_tokenizer_inspection_preview,
    load_training_sequences_for_tokenizer,
    save_inspection_preview,
)

# Reload the saved tokenizer from disk so the sanity check uses the same files
# that later notebooks will load.
loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(str(tokenizer_output_dir))
train_sequences = load_training_sequences_for_tokenizer(train_data_path)

# Build a compact inspection table that shows whether donor-side units stay
# grouped while acceptor-side units remain separate on real examples.
inspection_df = build_tokenizer_inspection_preview(
    tokenizer=loaded_tokenizer,
    sequences=train_sequences,
    num_samples=3,
    max_display_tokens=30,
)

display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')

if SAVE_INSPECTION_PREVIEW:
    saved_inspection_path = save_inspection_preview(
        inspection_df=inspection_df,
        output_path=inspection_preview_path,
    )
    print(f'Inspection preview saved to: {saved_inspection_path}')
else:
    print('Inspection preview saving is disabled for this run.')
